In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

In [2]:
ano = 2023

In [3]:
data_dir = Path("data")
INDIR = Path(f"../data/data_model/{ano}")
OUTDIR_IMG = Path(f"../report/img/{ano}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [4]:
arquivo_uf = INDIR / f"ANALISE_NOTAS_ENEM_UF_BRASIL_CLUSTERS_{ano}.csv"
df_uf = pd.read_csv(arquivo_uf, sep=",")

dict_uf = {
    0: df_uf[df_uf['CLUSTER'] == 0].copy().reset_index(drop=True),
    1: df_uf[df_uf['CLUSTER'] == 1].copy().reset_index(drop=True),
    2: df_uf[df_uf['CLUSTER'] == 2].copy().reset_index(drop=True)
}

In [5]:
df_uf.head()

,UF,QTD_PARTICIPANTES,RENDA_FAMILIAR_SM_MEDIA,NOTA_CN_MEDIA,NOTA_CH_MEDIA,NOTA_LC_MEDIA,NOTA_MT_MEDIA,NOTA_REDACAO_MEDIA,NOTA_GERAL_MEDIA,CLUSTER
0,AC,14978.0,1.312373,475.982201,508.341034,506.466277,494.253205,592.575778,515.523699,0
1,AL,54546.0,1.186398,482.897015,507.380635,502.369030,510.959392,607.799289,522.281072,0
2,AM,47486.0,1.129887,473.471926,499.428276,498.251855,487.380883,552.554016,502.217391,0
3,AP,18032.0,1.354142,476.215911,505.324518,499.754869,484.324795,583.027950,509.729608,0
4,BA,218630.0,1.161243,486.104492,514.033551,507.184744,508.756376,614.699081,526.155649,1


In [6]:

url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"


In [7]:
colunas_notas = [c for c in df_uf.columns if c.endswith("_MEDIA")]
cores_cluster = {0: "#ff0e0e", 1: "#1f77b4", 2: "#2ca02c"}

cores_desempenho = {
    "Baixo": cores_cluster[0],
    "Intermediário": cores_cluster[1],
    "Alto": cores_cluster[2],
}

In [8]:
df_map = df_uf.copy()
desempenho_map = {0: "Baixo", 1: "Intermediário", 2: "Alto"}
df_map["DESEMPENHO"] = df_map["CLUSTER"].map(desempenho_map)

fig = px.choropleth(
    df_map,
    geojson=url,
    locations="UF",
    featureidkey="properties.sigla",
    color="DESEMPENHO",
    color_discrete_map=cores_desempenho,
    category_orders={"DESEMPENHO": ["Baixo", "Intermediário", "Alto"]},
    title=f"Desempenho por estado (ENEM {ano})",
    labels={"DESEMPENHO": "Desempenho"},
    custom_data=[
        "UF",
        "CLUSTER",
        "NOTA_GERAL_MEDIA",
        "RENDA_FAMILIAR_SM_MEDIA",
        "QTD_PARTICIPANTES"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Estado: %{customdata[0]}<br>"
        "Cluster: %{customdata[1]}<br>"
        "Média Geral: %{customdata[2]:.2f}<br>"
        "Renda média: %{customdata[3]:.2f}<br>"
        "Participantes: %{customdata[4]}<br>"
        "<extra></extra>"
    )
)

fig.show()

In [9]:
df_map = df_uf.copy()

fig = px.choropleth(
    df_map,
    geojson=url,
    locations='UF',
    featureidkey='properties.sigla',
    color='NOTA_GERAL_MEDIA',
    color_continuous_scale='RdYlGn',
    title=f'Rendimento médio geral por estado (ENEM {ano})',
    labels={'NOTA_GERAL_MEDIA': 'Nota Geral Média'},
    custom_data=[
        'UF',
        'NOTA_GERAL_MEDIA',
        'RENDA_FAMILIAR_SM_MEDIA',
        'QTD_PARTICIPANTES'
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Estado: %{customdata[0]}<br>"
        "Nota Geral Média: %{customdata[1]:.2f}<br>"
        "Renda média familiar: %{customdata[2]:.2f}<br>"
        "Participantes: %{customdata[3]}<br>"
        "<extra></extra>"
    )
)
"""
caminho = OUTDIR_IMG / f"RENDIMENTO_GERAL_{ano}.png"
fig.write_image(str(caminho), scale=2)
print(f"Imagem salva em: {caminho}")
"""
fig.show()
